# The 2LPT upgrade: a sharper field than Zel'dovich

In Lagrangian perturbation theory, a particle starts at a grid point **q** and ends at **x** = **q** + **Ψ**, where **Ψ** is a displacement. Zel'dovich (1LPT) keeps only the *first-order* displacement. **2LPT adds the second-order term**, sourced by the tidal field through

$$ \nabla^2\phi^{(2)} = \sum_{i<j}\left[\phi^{(1)}_{,ii}\,\phi^{(1)}_{,jj} - \big(\phi^{(1)}_{,ij}\big)^2\right] $$

— the same Hessian of the potential the T-web classifier uses. Physically, this extra piece pulls mass *harder* into the forming structures: voids empty out more, knots get denser, and the density PDF gains the positive skewness (the tree-level value 34/7) that Zel'dovich underestimates.

Two honest notes. First, the large-scale web skeleton is similar between the two — the clearest difference shows up in the **density PDF**, not the web map. Second, both are *perturbative*: they're accurate in the quasi-linear regime but break down once particles cross (true shell-crossing needs N-body, i.e. real QUIJOTE). 2LPT is the cheap accuracy bump that's still pure NumPy and runs locally.

This notebook builds one generator that does both orders, runs them from the *same* linear field, and compares them.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from numpy.fft import fftn, ifftn, fftfreq

# --- config ---
N         = 128      # grid per side (drop to 96 for ~2x faster)
L         = 250.0    # box, Mpc/h
sigma_lin = 1.8      # linear-density rms: how nonlinear, and how strong the 2LPT correction
R_smooth  = 3.0      # Gaussian smoothing (Mpc/h) before classifying the web
lam_th    = 0.3      # tidal-tensor eigenvalue threshold
seed      = 42
CLASS_NAMES  = ['void', 'sheet', 'filament', 'knot']
CLASS_COLORS = ['#08306b', '#6baed6', '#fd8d3c', '#cb181d']

## 1. One generator, both orders

`lpt_delta(..., order=1)` is Zel'dovich; `order=2` adds the second-order displacement. The only new code over the Zel'dovich notebook is the block that builds the second-order source `S2` from the six second derivatives of the first-order potential.

In [ ]:
def Pk_bbks(k, ns=0.96, G=0.2):
    k = np.asarray(k, float); q = np.where(k > 0, k/G, 1e-10)
    T = (np.log(1+2.34*q)/(2.34*q)) * (1+3.89*q+(16.1*q)**2+(5.46*q)**3+(6.71*q)**4)**(-0.25)
    return k**ns * T**2

def cic(pos, N, L):
    g = np.zeros((N, N, N)); s = (pos % L)/(L/N); i = np.floor(s).astype(int) % N; fr = s - np.floor(s)
    for dx in (0, 1):
        for dy in (0, 1):
            for dz in (0, 1):
                wx = fr[:,0] if dx else 1-fr[:,0]; wy = fr[:,1] if dy else 1-fr[:,1]; wz = fr[:,2] if dz else 1-fr[:,2]
                ii=(i[:,0]+dx)%N; jj=(i[:,1]+dy)%N; kk=(i[:,2]+dz)%N
                np.add.at(g, (ii,jj,kk), wx*wy*wz)
    return g

def lpt_delta(N, L, sigma_lin=1.8, seed=42, order=2):
    kax = fftfreq(N, d=L/N)*2*np.pi; KX,KY,KZ = np.meshgrid(kax,kax,kax,indexing='ij')
    Kv = [KX,KY,KZ]; k2 = KX**2+KY**2+KZ**2; k2[0,0,0] = 1.0
    rng = np.random.default_rng(seed)
    dk = fftn(rng.standard_normal((N,N,N)))*np.sqrt(Pk_bbks(np.sqrt(k2))); dk[0,0,0] = 0
    d = ifftn(dk).real; d *= sigma_lin/d.std(); dk = fftn(d)        # fix linear-delta rms

    source_k = dk.copy()                                            # 1LPT source = linear delta
    if order == 2:
        pij = lambda a, b: ifftn(Kv[a]*Kv[b]/k2 * dk).real         # phi^(1)_,ij  (Hessian of potential)
        pxx,pyy,pzz = pij(0,0), pij(1,1), pij(2,2)
        pxy,pxz,pyz = pij(0,1), pij(0,2), pij(1,2)
        S2 = pxx*pyy + pxx*pzz + pyy*pzz - pxy**2 - pxz**2 - pyz**2 # 2nd-order source
        source_k = dk + (3.0/7.0)*fftn(S2)                          # EdS coefficient 3/7

    disp = np.array([ifftn(1j*Kv[i]*source_k/k2).real for i in range(3)])   # Psi = grad(potential)
    q = np.arange(N)*(L/N); QX,QY,QZ = np.meshgrid(q,q,q,indexing='ij'); Q = [QX,QY,QZ]
    pos = np.stack([(Q[i]+disp[i]).ravel() for i in range(3)], axis=1)
    rho = cic(pos, N, L)
    return rho/rho.mean() - 1.0

def smooth(d, L, R):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    return ifftn(fftn(d)*np.exp(-0.5*(KX**2+KY**2+KZ**2)*R**2)).real

def tweb(d, L, lam=0.3):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0; dk=fftn(d); T=np.zeros((N,N,N,3,3))
    for i in range(3):
        for j in range(i,3):
            t=ifftn((Kv[i]*Kv[j]/k2)*dk).real; T[...,i,j]=t
            if i!=j: T[...,j,i]=t
    return np.sum(np.linalg.eigvalsh(T) > lam, -1).astype(int)

print("generator ready")

## 2. Same linear field, two orders

Both fields use the same seed, so any difference is purely the second-order term.

In [ ]:
def skew(d): x = d - d.mean(); return (x**3).mean()/x.std()**3
def fracs(lab): f = np.bincount(lab.ravel(), minlength=4); return (100*f/f.sum()).round(1)

dZA = lpt_delta(N, L, sigma_lin, seed, order=1)
d2L = lpt_delta(N, L, sigma_lin, seed, order=2)
labZA = tweb(smooth(dZA, L, R_smooth), L, lam_th)
lab2L = tweb(smooth(d2L, L, R_smooth), L, lam_th)

print(f"Zel'dovich : skew {skew(smooth(dZA,L,2.0)):.2f}   V/S/F/K % {fracs(labZA)}")
print(f"2LPT       : skew {skew(smooth(d2L,L,2.0)):.2f}   V/S/F/K % {fracs(lab2L)}")
print("-> 2LPT: higher skew, emptier voids, more filament/knot")

## 3. Compare

Density slices, the cosmic-web slices, and the density PDF. Watch the PDF panel — that's where 2LPT clearly departs: an emptier void side (left) and a heavier knot tail (right).

In [ ]:
cmap = ListedColormap(CLASS_COLORS); norm = BoundaryNorm([-.5,.5,1.5,2.5,3.5], 4)
z = N//2
fig = plt.figure(figsize=(15, 10))

def dens(ax, d, t):
    ax.imshow(np.log10(1+d[z].T+1e-3), origin='lower', cmap='inferno', vmin=-1.5, vmax=1.0, extent=[0,L,0,L])
    ax.set_title(t); ax.set_xlabel('Mpc/h')
def web(ax, l, t):
    ax.imshow(l[z].T, origin='lower', cmap=cmap, norm=norm, extent=[0,L,0,L]); ax.set_title(t); ax.set_xlabel('Mpc/h')

dens(fig.add_subplot(2,3,1), dZA, "Zel'dovich (1LPT)  density")
dens(fig.add_subplot(2,3,2), d2L, "2LPT  density")

a3 = fig.add_subplot(2,3,3); b = np.linspace(-1.2, 1.5, 60)
a3.hist(np.log10(1+dZA.ravel()+1e-3), bins=b, histtype='step', color='#377eb8', lw=2, label="Zel'dovich", density=True)
a3.hist(np.log10(1+d2L.ravel()+1e-3), bins=b, histtype='step', color='#e41a1c', lw=2, label="2LPT", density=True)
a3.set_yscale('log'); a3.set_xlabel(r'$\log_{10}(1+\delta)$'); a3.set_title('density PDF'); a3.legend()

web(fig.add_subplot(2,3,4), labZA, "Zel'dovich  cosmic web")
web(fig.add_subplot(2,3,5), lab2L, "2LPT  cosmic web")

a6 = fig.add_subplot(2,3,6); im = a6.imshow(lab2L[z].T, origin='lower', cmap=cmap, norm=norm); a6.set_visible(False)
cb = fig.colorbar(im, ax=a6, ticks=[0,1,2,3], fraction=0.3); cb.ax.set_yticklabels(CLASS_NAMES)
plt.tight_layout(); plt.show()

## Using 2LPT everywhere, and what's next

- **Plug it into the U-Net.** In the U-Net notebook's `make_cube`, swap the Zel'dovich call for `lpt_delta(N, L, sigma_lin, seed, order=2)`. A crisper, more realistic target sharpens the inference task — exactly the upgrade we wanted before retraining.
- **The `sigma_lin` knob** controls both how nonlinear the field is and how large the 2LPT correction is. Push it up for richer structure (and a bigger 2LPT effect), but remember that far into the nonlinear regime both LPT orders break down.
- **For genuinely nonlinear knots**, the honest next step is N-body — i.e. a real QUIJOTE field via Route A, or running Pylians' tooling once the WSL env has it. 2LPT closes part of the gap to N-body cheaply, but not all of it.